In [1]:
from huggingface_hub import login
login() 

In [2]:
# Configuración de carpetas para entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent
BASE_DIR.mkdir(exist_ok=True)

for sub in ["context", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Estructura creada (si no existía):")
for p in ["context", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())

# Nota: el dataset debe existir en: CORPUS/proccesed data/wuxia_zh_en_clean


Base: C:\Users\Usuario\Desktop\TFG\CORPUS
Estructura creada (si no existía):
 - C:\Users\Usuario\Desktop\TFG\CORPUS\context
 - C:\Users\Usuario\Desktop\TFG\CORPUS\models
 - C:\Users\Usuario\Desktop\TFG\CORPUS\proccesed data


In [3]:
from dataclasses import dataclass

@dataclass
class Config:
    # Rutas (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"  
    output_dir: Path   = BASE_DIR / "models" / "llama3"
       
    evaluation_dir: Path = BASE_DIR / "evaluation"
    translate_dir: Path = BASE_DIR / "evaluation" / "translate" / "llm"
    translate_file: Path =   "llama3.txt"
    results_file: Path = "llm_results.txt"
    

    # Modelo
    # model_ckpt: str = "Qwen/Qwen-7B-Chat"
    
    model_ckpt: str= "meta-llama/Meta-Llama-3-8B-Instruct"
cfg = Config()
print(cfg)


Config(dataset_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/processed_data/wuxia_zh_en_clean'), output_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/models/llama3'), evaluation_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/evaluation'), translate_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/evaluation/translate/llm'), translate_file='llama3.txt', results_file='llm_results.txt', model_ckpt='meta-llama/Meta-Llama-3-8B-Instruct')


In [4]:
import random, numpy as np, torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [5]:
from datasets import load_from_disk


raw_ds = load_from_disk(cfg.dataset_dir)
print(raw_ds)

DatasetDict({
    train: Dataset({
        features: ['zh', 'en'],
        num_rows: 417208
    })
    validation: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
    test: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
})


In [6]:
example = raw_ds["train"][0]
print("ZH:", example["zh"])
print("EN:", example["en"])

ZH: 第章 听到白小纯的话语，看到圣皇的迟疑，邪皇这里顿时呼吸一促，他目中刹那就露出凌厉之芒，右手猛的一挥，顿时那残破的红日，骤然幻化
EN: chapter- The Saint-Emperor hesitated, and the Vile-Emperor sucked in a breath, eyes flickering with cold light as he waved his hand to summon his damaged red sun


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


tokenizer = AutoTokenizer.from_pretrained(cfg.model_ckpt)
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_ckpt,
    dtype=torch.bfloat16,
    device_map="auto",
)

# (Opcional) Verificamos si se ha cargado en GPU
device = model.device
print("Modelo cargado en dispositivo:", device)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Modelo cargado en dispositivo: cuda:0


In [8]:
# Calcular número de parámetros (opcionales, puede tardar un poco)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros totales del modelo: {total_params:,}")

Parámetros totales del modelo: 8,030,261,248


In [9]:
def translate_with_prompt(chinese_text, system_msg, user_instruction, shots=None, max_new_tokens=200):
    
    if shots:
        examples = "\n".join(
            f"Example {i}\nChinese: {ex['zh']}\nEnglish: {ex['en']}\n"
            for i, ex in enumerate(shots, 1)
        )
        system_msg = (
            f"{system_msg}\n\n"
            f"Below are example translations to guide your style:\n"
            f"{examples}\n"
            "Use these examples for reference, but DO NOT repeat them or mention them in your output."
        )

    # Mensaje del usuario: solo la tarea actual
    user_msg = (
        f"{user_instruction}\n"
        f"Convert all Chinese personal names, place names, and martial arts terms into pinyin romanization (do not translate them into English). Respond ONLY with the English translation:\n"
        f"Chinese: {chinese_text}\nEnglish:"
    )

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]

    # Puede devolver un tensor (input_ids) o un BatchEncoding
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to(model.device)

    if isinstance(inputs, torch.Tensor):
        input_ids = inputs
        attention_mask = torch.ones_like(input_ids)
    else:
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]

    gen = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    
    new_tokens = gen[:, input_ids.shape[1]:]
    output_text = tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()
    return output_text


In [10]:
prompts = [
    {
        "system_msg": (
            "Your role will be that of an assistant that translates Chinese wuxia text into fluent English prose."
            "Your translations should read as if they were written by a native English novelist while preserving the atmosphere and meaning of ancient martial worlds."
            "Avoid literal phrasing and focus on emotional tone and narrative flow. "
            "Always enclose your final English translation between triple square brackets [[[ and ]]] so it can be easily identified."
        ),
        "user_instruction": (
            "Translate the following Chinese text into natural English, making it smooth and stylistically consistent with modern fantasy novels:"
        ),
    },
    {
        "system_msg": (
            "You are an expert literary translator of Chinese wuxia fiction. "
            "Retain the poetic flow, cultural symbolism, and epic atmosphere characteristic of wuxia narratives. "
            "The goal is to produce English reflecting the depth of Chinese idioms. "
            "Surround the complete translated text with [[[ and ]]] to clearly mark the translation."
        ),
        "user_instruction": (
            "Translate the following Chinese passage into refined, literary English that conveys the spirit of classic wuxia storytelling:"
        ),
    },
    {
        "system_msg": (
            "You are functioning as a seq2seq neural translation model trained for Chinese-to-English tasks. "
            "You must produce a one-to-one translation of the source text without commentary, explanation, or rephrasing. "
            "Do not include introductions or stylistic variations, just output the translation. "
            "The translated text must be enclosed between [[[ and ]]] for clear identification."
        ),
        "user_instruction": (
            "Provide a direct English translation of the following Chinese text, maintaining word order and fidelity as much as possible, similar to a seq2seq model:"
        ),
    },
    {
        "system_msg": (
            "You are a bilingual scholar translating Chinese martial arts literature into English. "
            "Your translation must be accurate, faithful, and formal. "
            "Do not include introductions or stylistic variations, just output the translation. "
            "Enclose the entire translation between [[[ and ]]] so it can be clearly extracted."
        ),
        "user_instruction": (
            "Translate the following text into precise English, maintaining original meanings and cultural nuances:"
        ),
    },
    {
        "system_msg": (
            "You are simulating a traditional sequence-to-sequence translation model working on chinese to english translations. "
            "Translate the input text directly into English without adding explanations, literary style, or rewording. "
            "Produce a literal, token-aligned translation suitable for machine translation evaluation. "
            "The output must be wrapped between [[[ and ]]] so it can be parsed automatically."
        ),
        "user_instruction": (
            "Translate the following Chinese text literally into English, preserving structure and meaning with minimal adaptation:"
        ),
    }
]


In [11]:
import os
import time
from tqdm import tqdm
from datetime import datetime


N = 100
PROMPT_IDS = [0, 1, 2, 3, 4]  # índices de los prompts a evaluar (por defecto 5 primeros)
SHOTS_CONFIG = {
    "0shot": 0,
    "1shot": 1,
    "2shot": 2,
    "5shot": 5,
}

output_dir = Path(cfg.translate_dir)
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = output_dir / f"{cfg.translate_file}_{timestamp}.txt"
time_log_file = output_dir / f"{cfg.translate_file}_{timestamp}_timing.csv"

print(f"Guardando resultados en: {output_file}")
print(f"Guardando tiempos en: {time_log_file}")

all_time_records = []




Guardando resultados en: c:\Users\Usuario\Desktop\TFG\CORPUS\evaluation\translate\llm\llama3.txt_20251027_171224.txt
Guardando tiempos en: c:\Users\Usuario\Desktop\TFG\CORPUS\evaluation\translate\llm\llama3.txt_20251027_171224_timing.csv


In [12]:

def build_shots(num_shots, split="train"):
    """Devuelve una lista con num_shots ejemplos del dataset de entrenamiento."""
    if num_shots == 0:
        return []
    total_train = len(raw_ds[split])
    return [raw_ds[split][i % total_train] for i in range(num_shots)]


def evaluate_prompt(prompt_id, n=N):
    """Ejecuta la evaluación completa de un prompt (todas las configuraciones de shot)."""
    prompt = prompts[prompt_id]
    system_msg = prompt["system_msg"]
    user_instruction = prompt["user_instruction"]

    # Contenedores para resultados
    log_lines = []
    time_records = []

    log_lines.append(f"\n\n##############################################")
    log_lines.append(f"### PROMPT {prompt_id}")
    log_lines.append(f"### {system_msg[:120]}...")
    log_lines.append(f"##############################################\n")

    prompt_total_time = 0.0 

    for mode, n_shots in SHOTS_CONFIG.items():
        log_lines.append(f"\n=== {mode.upper()} ({n_shots}-shot) ===\n")
        mode_times = []

        for i in tqdm(range(n), desc=f"Prompt {prompt_id} -> {mode}"):
            sample = raw_ds["test"][i % len(raw_ds["test"])]
            
            if n_shots == 0:
                shots = None 
            else:
                shots = build_shots(n_shots)
            start_time = time.perf_counter()
            try:
                translation = translate_with_prompt(
                    sample["zh"],
                    system_msg=system_msg,
                    user_instruction=user_instruction,
                    shots=shots
                )
            except Exception as e:
                translation = f"[Error en iteración {i}: {e}]"
            elapsed = time.perf_counter() - start_time
            mode_times.append(elapsed)

            # Guardar en texto
            log_lines.append(f"--- Caso {i+1} ---")
            log_lines.append(f"[ZH]: {sample['zh']}")
            log_lines.append(f"[REF]: {sample['en']}")
            log_lines.append(f"[GEN]: {translation.strip()}")
            log_lines.append(f"[TIME]: {elapsed:.2f} s\n")

        avg_time = sum(mode_times) / len(mode_times)
        total_time = sum(mode_times)
        prompt_total_time += total_time
        time_records.append((prompt_id, mode, n_shots, avg_time, total_time))

        log_lines.append(f"\n>>> Tiempo medio {mode}: {avg_time:.2f} s | Total: {total_time:.1f} s\n")

    # resumen total por prompt
    log_lines.append(f"\n##### TOTAL PROMPT {prompt_id}: {prompt_total_time/60:.2f} min #####\n")
    return "\n".join(log_lines), time_records


In [13]:
with open(output_file, "w", encoding="utf-8") as f_out:
    for pid in PROMPT_IDS:
        block_text, time_data = evaluate_prompt(pid, n=N)
        f_out.write(block_text)
        f_out.flush()
        all_time_records.extend(time_data)



Prompt 0 -> 0shot:   0%|          | 0/100 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Prompt 4 -> 5shot: 100%|██████████| 100/100 [32:57<00:00, 19.78s/it]


In [14]:

import csv
with open(time_log_file, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["prompt_id", "mode", "n_shots", "avg_time_s", "total_time_s"])
    writer.writerows(all_time_records)

# Resumen general
summary = {}
for pid, mode, n_shots, avg_t, total_t in all_time_records:
    summary.setdefault(pid, 0)
    summary[pid] += total_t

print("\n=== RESUMEN TIEMPOS TOTALES (por prompt) ===")
for pid, total in summary.items():
    print(f"Prompt {pid}: {total/60:.2f} min")

print(f"\nEvaluación completada.\nResultados: {output_file}\nTiempos: {time_log_file}")


=== RESUMEN TIEMPOS TOTALES (por prompt) ===
Prompt 0: 166.17 min
Prompt 1: 211.39 min
Prompt 2: 145.19 min
Prompt 3: 149.04 min
Prompt 4: 143.53 min

Evaluación completada.
Resultados: c:\Users\Usuario\Desktop\TFG\CORPUS\evaluation\translate\llm\llama3.txt_20251027_171224.txt
Tiempos: c:\Users\Usuario\Desktop\TFG\CORPUS\evaluation\translate\llm\llama3.txt_20251027_171224_timing.csv
